# visualize

> Look at the data and the model: verify image/mask alignment, overlay predictions on inputs, and surface the worst patches for error analysis.

Good segmentation work is mostly looking at pictures. `sanity_check` confirms augmentations keep masks aligned; `show_pred` overlays ground truth vs prediction; `show_top_losses` is the fastest way to find label errors and hard cases.

In [ ]:
#| default_exp visualize

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch

from segmentation_tools_pt.inference import predict, decode
from segmentation_tools_pt.losses import SegLoss
from segmentation_tools_pt.metrics import SegMetrics

## Overlay one prediction

Ground-truth-only pixels are green, prediction-only red, agreement magenta. Foreground from all classes is grouped together, so the same view works for binary and multi-class. `pred` may be a label map or a `(C, H, W)` probability map.

In [ ]:
#| export
def _to_numpy(x):
    "Detach a tensor to numpy; pass arrays through."
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

def show_pred(
    image,                   # (C, H, W) or (H, W, C) input
    mask,                    # (H, W) integer-label ground truth
    pred,                    # (H, W) labels or (C, H, W) probabilities
    threshold: float = 0.5,  # binary decision threshold if pred is probabilities
    ax=None,                 # matplotlib Axes to draw on
):
    """Overlay ground truth vs prediction on the first image channel. Returns the Axes."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    image = _to_numpy(image)
    if image.ndim == 3 and image.shape[0] <= 4:
        image = image.transpose(1, 2, 0)
    mask = _to_numpy(mask).squeeze()
    pred = _to_numpy(pred).squeeze()
    if pred.ndim == 3:
        pred = (pred[0] >= threshold).astype(int) if pred.shape[0] == 1 else pred.argmax(0)
    gt_fg, pr_fg = mask > 0, pred > 0
    overlay = np.zeros((*gt_fg.shape, 3), np.float32)
    overlay[gt_fg & ~pr_fg] = (0.0, 0.8, 0.0)
    overlay[pr_fg & ~gt_fg] = (0.8, 0.0, 0.0)
    overlay[gt_fg & pr_fg] = (1.0, 0.0, 1.0)
    disp = image[..., 0] if image.ndim == 3 else image
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=0.5); ax.set_axis_off()
    ax.legend(handles=[mpatches.Patch(color=(0, 0.8, 0), label='GT'),
                       mpatches.Patch(color=(0.8, 0, 0), label='Pred'),
                       mpatches.Patch(color=(1, 0, 1), label='Both')], loc='upper right', fontsize=7)
    return ax

## Headless save helpers

YOLO-style validation JPEGs and fastai-style activation histogram PNGs for training callbacks. Uses the Agg backend so saves work without a display.

In [ ]:
#| export
def _disp_channel(image):
    "First channel of a (C,H,W) or (H,W) image for grayscale display."
    image = _to_numpy(image)
    if image.ndim == 3 and image.shape[0] <= 4:
        image = image.transpose(1, 2, 0)
    return image[..., 0] if image.ndim == 3 else image

def _pred_labels(pred, threshold: float = 0.5):
    pred = _to_numpy(pred).squeeze()
    if pred.ndim == 3:
        return (pred[0] >= threshold).astype(int) if pred.shape[0] == 1 else pred.argmax(0)
    return pred

def overlay_gt_only(image, mask, ax=None):
    """Green GT overlay on the first image channel (YOLO `*_labels.jpg` style)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    disp = _disp_channel(image)
    gt_fg = _to_numpy(mask).squeeze() > 0
    overlay = np.zeros((*gt_fg.shape, 3), np.float32)
    overlay[gt_fg] = (0.0, 0.8, 0.0)
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=0.5); ax.set_axis_off()
    return ax

def overlay_pred_only(image, pred, threshold: float = 0.5, ax=None):
    """Red prediction overlay on the first image channel (YOLO `*_pred.jpg` style)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    disp = _disp_channel(image)
    pr_fg = _pred_labels(pred, threshold) > 0
    overlay = np.zeros((*pr_fg.shape, 3), np.float32)
    overlay[pr_fg] = (0.8, 0.0, 0.0)
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=0.5); ax.set_axis_off()
    return ax

def save_color_dim_png(hist_stack, path: str | Path, title: str = '') -> None:
    """Save fastai-style colorful-dimension plot: log1p(hist) imshow, bins on y, batches on x."""
    h = _to_numpy(hist_stack).astype(np.float64)
    if h.ndim == 1:
        h = h.reshape(1, -1)
    if h.shape[0] > h.shape[1]:
        h = h.T
    res = np.log1p(h)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(res, origin='lower', aspect='auto', cmap='viridis')
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=9)
    fig.savefig(path, bbox_inches='tight', dpi=100)
    plt.close(fig)

def save_near_zero_png(epochs: List[int], values: List[float], path: str | Path, title: str = '') -> None:
    "Line plot of near-zero activation fraction over epochs."
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(epochs, values, marker='o', markersize=3)
    ax.set_xlabel('epoch'); ax.set_ylabel('% near zero'); ax.set_ylim(0, 1)
    if title:
        ax.set_title(title, fontsize=9)
    fig.savefig(path, bbox_inches='tight', dpi=100)
    plt.close(fig)

def save_overlay_grid_jpeg(
    images,
    masks,
    preds,
    path: str | Path,
    threshold: float = 0.5,
    mode: str = 'compare',
    jpeg_quality: int = 90,
    max_cols: int = 4,
) -> None:
    """Mosaic batch overlays and save as JPEG (`labels`, `pred`, or `compare`)."""
    imgs = _to_numpy(images)
    msks = _to_numpy(masks)
    prds = _to_numpy(preds)
    if imgs.ndim == 3:
        imgs, msks, prds = imgs[None], msks[None], prds[None]
    n = imgs.shape[0]
    rows, cols = _grid(n, min(max_cols, n))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.atleast_1d(axes).flatten()
    for i in range(n):
        img, msk, prd = imgs[i], msks[i], prds[i]
        if mode == 'labels':
            overlay_gt_only(img, msk, ax=axes[i])
        elif mode == 'pred':
            overlay_pred_only(img, prd, threshold, ax=axes[i])
        else:
            show_pred(img, msk, prd, threshold, ax=axes[i])
    for i in range(n, len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    fig.savefig(path, format='jpeg', dpi=100, pil_kwargs={'quality': jpeg_quality}, bbox_inches='tight')
    plt.close(fig)

## Sanity-check the dataset

Before training, plot a few augmented samples to confirm the mask still lines up with the image after geometric transforms.

In [ ]:
#| export
def _grid(n: int, cols: int) -> Tuple[int, int]:
    "Rows/cols for `n` panels."
    return math.ceil(n / cols), cols

def sanity_check(dataset, n: int = 8):
    "Plot `n` (image, mask) pairs to verify alignment."
    n = min(n, len(dataset))
    rows, cols = _grid(n, min(5, n))
    fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 5, rows * 3))
    axes = np.array(axes).reshape(-1, 2)
    for i in range(n):
        s = dataset[i]
        img = _to_numpy(s['image'])
        img = img[0] if img.ndim == 3 else img
        axes[i, 0].imshow(img, cmap='gray'); axes[i, 0].set_title(str(s['sample_id']), fontsize=7)
        axes[i, 1].imshow(_to_numpy(s['mask']).squeeze(), cmap='gray')
        for a in axes[i]:
            a.set_axis_off()
    for i in range(n, len(axes)):
        for a in axes[i]:
            a.set_visible(False)
    plt.tight_layout()
    return fig

## Predictions and worst cases

In [ ]:
#| export
def show_batch(model, dataset, device, n: int = 8, threshold: float = 0.5):
    "Overlay predictions for `n` random samples."
    idxs = np.random.choice(len(dataset), min(n, len(dataset)), replace=False)
    rows, cols = _grid(len(idxs), min(4, len(idxs)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).flatten()
    for i, idx in enumerate(idxs):
        s = dataset[int(idx)]
        probs = predict(model, s['image'], device)
        show_pred(s['image'], s['mask'], decode(probs, threshold), threshold, ax=axes[i])
        axes[i].set_title(str(s['sample_id']), fontsize=8)
    for i in range(len(idxs), len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    return fig

from contextlib import contextmanager

@contextmanager
def _eval_mode(model):
    "Put `model` in eval mode, then restore each submodule's own train/eval flag (frozen BatchNorm stays frozen) even if the body raises."
    flags = [(m, m.training) for m in model.modules()]
    model.eval()
    try:
        yield
    finally:
        for m, was_training in flags:
            m.training = was_training

@torch.no_grad()
def compute_patch_losses(model, dataset, criterion, device, n: Optional[int] = None) -> List[tuple]:
    "Per-sample loss, sorted worst-first as `(idx, loss, sample_id)`."
    n = n or len(dataset)
    results = []
    with _eval_mode(model):
        for idx in range(min(n, len(dataset))):
            s = dataset[idx]
            logits = model(s['image'].unsqueeze(0).to(device))
            loss = float(criterion(logits, s['mask'].unsqueeze(0).to(device))['loss'])
            results.append((idx, loss, s['sample_id']))
    results.sort(key=lambda r: r[1], reverse=True)
    return results

def show_top_losses(model, dataset, criterion, device, k: int = 8, threshold: float = 0.5):
    "Overlay the `k` highest-loss samples for error analysis."
    top = compute_patch_losses(model, dataset, criterion, device)[:k]
    rows, cols = _grid(len(top), min(4, len(top)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).flatten()
    for i, (idx, loss, sid) in enumerate(top):
        s = dataset[idx]
        show_pred(s['image'], s['mask'], decode(predict(model, s['image'], device), threshold), threshold, ax=axes[i])
        axes[i].set_title(f'{sid}\nloss={loss:.4f}', fontsize=7)
    for i in range(len(top), len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    return fig

## Multilabel overlay

`show_pred` groups all foreground together (green/red/magenta), which loses
information once there's more than one class -- for chip/void/lifted you need
to see *which* class is which, and where they nest (void inside chip). Each
mask channel gets its own color, drawn in channel order: later channels paint
over earlier ones, so a nested region shows through as its own color rather
than blending into an ambiguous mix.

In [ ]:
#| export
DEFAULT_MULTILABEL_COLORS = [
    (0.9, 0.1, 0.1), (0.1, 0.8, 0.1), (0.2, 0.4, 0.95),
    (0.9, 0.8, 0.1), (0.8, 0.1, 0.9), (0.1, 0.8, 0.8),
]

def overlay_multilabel(
    image,                # (C, H, W) or (H, W, C) input
    masks,                 # (K, H, W) binary/bool -- one channel per class
    class_names=None,      # K names for the legend; defaults to '0', '1', ...
    colors=None,            # K colors; defaults to DEFAULT_MULTILABEL_COLORS, cycled
    alpha: float = 0.5,     # overlay opacity
    ax=None,                # matplotlib Axes to draw on
):
    """Tint each mask channel with its own color, drawn in channel order so nested
    regions (e.g. void inside chip) stay visually distinct rather than blending."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    disp = _disp_channel(image)
    masks = _to_numpy(masks)
    if masks.ndim == 2:
        masks = masks[None]
    k = masks.shape[0]
    colors = colors or DEFAULT_MULTILABEL_COLORS
    class_names = class_names or [str(i) for i in range(k)]
    overlay = np.zeros((*masks.shape[1:], 3), np.float32)
    for i in range(k):
        overlay[masks[i] > 0.5] = colors[i % len(colors)]
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=alpha); ax.set_axis_off()
    handles = [mpatches.Patch(color=colors[i % len(colors)], label=class_names[i]) for i in range(k)]
    ax.legend(handles=handles, loc='upper right', fontsize=7)
    return ax

def show_pred_multilabel(
    image,                 # (C, H, W) or (H, W, C) input
    gt_masks,               # (K, H, W) ground truth, binary/bool
    pred_masks,              # (K, H, W) prediction -- probabilities or binary/bool
    class_names=None,       # K names for the legend
    threshold=0.5,           # decision threshold if pred_masks looks like probabilities
    colors=None,
    axes=None,               # a length-2 array of Axes (GT, Pred); created if None
):
    """Side-by-side GT vs Pred multilabel overlay using the same per-channel colors."""
    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=(10, 5))
    pr = _to_numpy(pred_masks)
    if np.issubdtype(pr.dtype, np.floating):
        c = pr.shape[0]
        # duck-type "sequence of per-channel thresholds" rather than isinstance(list, tuple):
        # Hydra/OmegaConf configs hand back ListConfig, not a plain list, for a YAML list.
        is_seq = hasattr(threshold, '__len__') and not isinstance(threshold, (str, bytes))
        t = list(threshold) if is_seq else [threshold] * c
        pr = np.stack([pr[i] >= t[i] for i in range(c)])
    overlay_multilabel(image, gt_masks, class_names, colors, ax=axes[0]); axes[0].set_title('GT')
    overlay_multilabel(image, pr, class_names, colors, ax=axes[1]); axes[1].set_title('Pred')
    return axes

def save_multilabel_overlay_grid_jpeg(
    images,
    gt_masks,
    pred_masks,
    path: str | Path,
    class_names=None,
    threshold=0.5,
    colors=None,
    jpeg_quality: int = 90,
    max_cols: int = 2,
) -> None:
    """Mosaic batch GT-vs-Pred multilabel overlays (2 panels per sample) and save as JPEG."""
    imgs = _to_numpy(images)
    gts = _to_numpy(gt_masks)
    prds = _to_numpy(pred_masks)
    if imgs.ndim == 3:
        imgs, gts, prds = imgs[None], gts[None], prds[None]
    n = imgs.shape[0]
    cols = min(max_cols, n)
    rows, _ = _grid(n, cols)
    fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 6, rows * 3))
    axes = np.array(axes).reshape(rows, cols * 2)
    for i in range(n):
        r, c = divmod(i, cols)
        show_pred_multilabel(imgs[i], gts[i], prds[i], class_names, threshold, colors,
                              axes=axes[r, c * 2:c * 2 + 2])
    for i in range(n, rows * cols):
        r, c = divmod(i, cols)
        for a in axes[r, c * 2:c * 2 + 2]:
            a.set_visible(False)
    plt.tight_layout()
    fig.savefig(path, format='jpeg', dpi=100, pil_kwargs={'quality': jpeg_quality}, bbox_inches='tight')
    plt.close(fig)

In [ ]:
#| test
# test: overlay_multilabel colors each channel, later channels drawn on top so nesting shows through
import matplotlib
matplotlib.use('Agg')
import tempfile
img = torch.rand(1, 8, 8)
chip = torch.zeros(8, 8); chip[1:7, 1:7] = 1
void = torch.zeros(8, 8); void[3:5, 3:5] = 1  # strictly inside chip
masks = torch.stack([chip, void])  # (2, 8, 8): 0=chip, 1=void

ax = overlay_multilabel(img, masks, class_names=['chip', 'void'])
assert ax is not None
rgba = ax.images[1].get_array()  # the overlay imshow, not the base grayscale
chip_only_px = tuple(np.array(rgba[1, 1][:3]))   # inside chip, outside void
void_px = tuple(np.array(rgba[3, 3][:3]))         # inside void (drawn after chip -> its own color)
assert chip_only_px != void_px  # visually distinguishable, not just "foreground"
assert chip_only_px == DEFAULT_MULTILABEL_COLORS[0]
assert void_px == DEFAULT_MULTILABEL_COLORS[1]

axes = show_pred_multilabel(img, masks, masks.bool(), class_names=['chip', 'void'])
assert len(axes) == 2  # GT panel + Pred panel

with tempfile.TemporaryDirectory() as d:
    imgs2 = torch.stack([img, img])
    gts2 = torch.stack([masks, masks])
    prds2 = torch.stack([masks.bool(), masks.bool()])
    save_multilabel_overlay_grid_jpeg(imgs2, gts2, prds2, Path(d) / 'ml_grid.jpg', class_names=['chip', 'void'])
    assert (Path(d) / 'ml_grid.jpg').exists()
print('overlay_multilabel / show_pred_multilabel / save_multilabel_overlay_grid_jpeg ✓  per-channel colors, nesting visible')

In [ ]:
# test: overlay, sanity check, headless saves, and worst-first ordering on a tiny dataset
import matplotlib
matplotlib.use('Agg')
import tempfile
import torch.nn as nn
from torch.utils.data import Dataset
from segmentation_tools_pt.losses import SegLoss

class _DS(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:10, 4:10] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': str(i)}

ds = _DS()
assert show_pred(ds[0]['image'], ds[0]['mask'], ds[0]['mask']) is not None
assert sanity_check(ds, n=2) is not None
res = compute_patch_losses(nn.Conv2d(3, 1, 3, padding=1), ds, SegLoss(), torch.device('cpu'))
assert len(res) == 4 and res[0][1] >= res[-1][1]
imgs = torch.stack([ds[i]['image'] for i in range(2)])
msks = torch.stack([ds[i]['mask'] for i in range(2)])
prds = msks.clone()
with tempfile.TemporaryDirectory() as d:
    save_color_dim_png(torch.rand(40, 3), Path(d) / 'color.png')
    save_near_zero_png([0, 1], [0.1, 0.2], Path(d) / 'near_zero.png')
    save_overlay_grid_jpeg(imgs, msks, prds, Path(d) / 'grid.jpg', mode='compare')
    assert (Path(d) / 'color.png').exists()
    assert (Path(d) / 'near_zero.png').exists()
    assert (Path(d) / 'grid.jpg').exists()

## Post-training error posters

`show_top_losses` overlays GT-vs-pred only; real triage needs to see *which kind* of error
the model makes. For the post-training `ErrorAnalysisCallback` (see `10_train.ipynb`, issue
#18) each ranked sample becomes one row of a 4-column JPEG poster:

1. **Image** — raw input, first/display channel.
2. **GT overlay** — ground truth tinted green at a low alpha (default 0.1, so the underlying
   X-ray/optical content stays readable).
3. **Pred overlay** — model prediction tinted red at the same alpha.
4. **Difference overlay** — FN green / FP red / overlap magenta, so misses vs false alarms
   are directly comparable.

### Step 1 — per-sample ranking (loss and pixel metrics)

In [ ]:
# experiment: per-sample ranking sketches inline (not exported yet) -- loss vs IoU on synthetic multilabel data
import matplotlib
matplotlib.use('Agg')
import torch.nn as nn
from torch.utils.data import Dataset
from segmentation_tools_pt.losses import SegLoss
from segmentation_tools_pt.metrics import SegMetrics

class _ToyML(Dataset):
    def __len__(self): return 5
    def __getitem__(self, i):
        m = torch.zeros(3, 16, 16)
        m[0, 2:8, 2:8] = 1.0; m[1, 9:14, 4:12] = 1.0
        m[2] = (m[1] * (torch.rand(16, 16) > 0.7)).float()  # nested class inside class 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': f'sample_{i}'}

toy, dev = _ToyML(), torch.device('cpu')
ml_model = nn.Conv2d(3, 3, 3, padding=1)
crit = SegLoss(task='multilabel')
proto = []
for i in range(len(toy)):
    s = toy[i]
    lg = ml_model(s['image'].unsqueeze(0))
    loss_score = float(crit(lg, s['mask'].unsqueeze(0))['loss'])
    mm = SegMetrics(num_classes=3, task='multilabel'); mm.update(lg, s['mask'].unsqueeze(0))
    iou_score = mm.compute()['iou']
    proto.append((i, loss_score, iou_score))
print('per-sample (idx, loss, iou):', [(i, round(l, 3), round(o, 3)) for i, l, o in proto])

In [ ]:
# verify: loss ranks worst-first (descending), IoU ranks worst-first (ascending); per-class loss via a binary criterion copy
loss_rank = sorted(proto, key=lambda r: r[1], reverse=True)
iou_rank = sorted(proto, key=lambda r: r[2])
assert loss_rank[0][1] >= loss_rank[-1][1] and iou_rank[0][2] <= iou_rank[-1][2]
crit_c1 = SegLoss(alpha=crit.tversky.alpha, beta=crit.tversky.beta, gamma=crit.tversky.gamma,
                  w_tversky=crit.w_tversky, w_dice=crit.w_dice, w_boundary=crit.w_boundary, task='binary')
s = toy[0]
lg = ml_model(s['image'].unsqueeze(0)); t = s['mask'].unsqueeze(0)
c1 = float(crit_c1(lg[:, 1:2], t[:, 1:2])['loss'])
print('class-1-only loss:', round(c1, 4))
assert c1 >= 0

In [ ]:
# experiment: one 4-column poster row drawn inline (image | GT | pred | difference)
row_fig, ax = plt.subplots(1, 4, figsize=(12, 3.4))
s = toy[0]
img = s['image'][0].numpy()
gt = s['mask'][0].numpy() > 0.5
lg = ml_model(s['image'].unsqueeze(0))
pr = (torch.sigmoid(lg)[0, 0] >= 0.5).numpy()
overlays = []
for fg, color in ((gt, (0.0, 0.8, 0.0)), (pr, (0.8, 0.0, 0.0))):
    ov = np.zeros((*fg.shape, 3), np.float32); ov[fg] = color
    overlays.append(ov)
df = np.zeros((*gt.shape, 3), np.float32)
df[gt & ~pr] = (0.0, 0.8, 0.0); df[pr & ~gt] = (0.8, 0.0, 0.0); df[gt & pr] = (1.0, 0.0, 1.0)
for a, base, ov, ttl in zip(ax, [img] * 4, [None, overlays[0], overlays[1], df],
                            ['image', 'GT', 'pred', 'diff']):
    a.imshow(base, cmap='gray'); a.set_axis_off()
    if ov is not None: a.imshow(ov, alpha=0.1)
    a.set_title(ttl, fontsize=8)
plt.tight_layout()

In [ ]:
# verify: the row saves as a real JPEG
import tempfile
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / 'row.jpg'
    row_fig.savefig(p, format='jpeg', dpi=100, pil_kwargs={'quality': 90}, bbox_inches='tight')
    plt.close(row_fig)
    assert p.exists() and p.stat().st_size > 0
    print('poster row jpeg bytes:', p.stat().st_size)

### Promote — export the working pieces

`rank_samples` / `rank_samples_per_class` return worst-first records; `save_error_poster`
draws the 4-column grid for the top-k records and saves it as a JPEG.

In [ ]:
#| export
_METRIC_KEYS = ('iou', 'dice', 'recall', 'precision')

def _check_rank_by(rank_by: str) -> str:
    "Reject unknown rank_by values early."
    assert rank_by == 'loss' or rank_by in _METRIC_KEYS, \
        f"rank_by must be 'loss' or one of {_METRIC_KEYS}, got {rank_by!r}"
    return rank_by

def _class_criterion(crit: SegLoss, class_idx: int) -> SegLoss:
    "Binary copy of a multilabel criterion, used to score ONE class channel independently."
    tv = crit.tversky
    return SegLoss(
        alpha=tv.alpha, beta=tv.beta, gamma=tv.gamma,
        w_tversky=crit.w_tversky, w_dice=crit.w_dice, w_boundary=crit.w_boundary,
        w_bce=crit.w_bce, task='binary',
    )

@torch.no_grad()
def rank_samples(
    model,                    # model returning (B, C, H, W) logits
    dataset,                  # Dataset of {'image', 'mask', 'sample_id'}
    criterion: SegLoss,       # loss returning a breakdown dict (rank_by='loss')
    device: torch.device,
    rank_by: str = 'loss',    # 'loss' (highest first) | 'iou' | 'dice' | 'recall' | 'precision' (lowest first)
    n: Optional[int] = None,  # rank at most n samples; None = whole dataset
    threshold=0.5,            # sigmoid threshold for metric ranking
    task: str = None,         # 'binary' | 'multiclass' | 'multilabel'; must be 'multilabel' for multilabel datasets
) -> List[dict]:
    "Per-sample ranking, worst-first, as `{'idx', 'sample_id', 'score'}` records."
    _check_rank_by(rank_by)
    n = n or len(dataset)
    records = []
    with _eval_mode(model):
        for idx in range(min(n, len(dataset))):
            s = dataset[idx]
            logits = model(s['image'].unsqueeze(0).to(device))
            if rank_by == 'loss':
                score = float(criterion(logits, s['mask'].unsqueeze(0).to(device))['loss'])
            else:
                m = SegMetrics(num_classes=logits.shape[1], task=task, threshold=threshold)
                m.update(logits, s['mask'].unsqueeze(0).to(device))
                score = m.compute()[rank_by]
            records.append({'idx': idx, 'sample_id': str(s['sample_id']), 'score': score})
    records.sort(key=lambda r: r['score'], reverse=rank_by == 'loss')
    return records

@torch.no_grad()
def rank_samples_per_class(
    model, dataset, criterion: SegLoss, device,
    class_idx: int,           # which multilabel class head to score
    rank_by: str = 'loss',
    n: Optional[int] = None,
    threshold=0.5,
) -> List[dict]:
    "Per-sample ranking for ONE multilabel class head, worst-first. Loss ranking slices the class channel and scores it with a binary copy of the criterion."
    _check_rank_by(rank_by)
    n = n or len(dataset)
    crit_c = _class_criterion(criterion, class_idx) if rank_by == 'loss' else None
    records = []
    with _eval_mode(model):
        for idx in range(min(n, len(dataset))):
            s = dataset[idx]
            logits = model(s['image'].unsqueeze(0).to(device))
            t = s['mask'].unsqueeze(0).to(device)
            if t.dim() == logits.dim() - 1:
                t = t.unsqueeze(1)
            ch_logits, ch_t = logits[:, class_idx:class_idx + 1], t[:, class_idx:class_idx + 1]
            if rank_by == 'loss':
                score = float(crit_c(ch_logits, ch_t)['loss'])
            else:
                m = SegMetrics(num_classes=1, task='binary', threshold=threshold)
                m.update(ch_logits, ch_t)
                score = m.compute()[rank_by]
            records.append({'idx': idx, 'sample_id': str(s['sample_id']), 'score': score})
    records.sort(key=lambda r: r['score'], reverse=rank_by == 'loss')
    return records

In [ ]:
#| export
_FN_COLOR, _FP_COLOR, _TP_COLOR = (0.0, 0.8, 0.0), (0.8, 0.0, 0.0), (1.0, 0.0, 1.0)

def _flat_overlay(image, fg, color, alpha: float, ax):
    "Single-color foreground overlay on the display channel, at a configurable alpha."
    fg = _to_numpy(fg).squeeze() > 0.5
    overlay = np.zeros((*fg.shape, 3), np.float32)
    overlay[fg] = color
    ax.imshow(_disp_channel(image), cmap='gray'); ax.imshow(overlay, alpha=alpha); ax.set_axis_off()
    return ax

def _diff_overlay(image, gt_fg, pr_fg, alpha: float, ax):
    "FN green / FP red / overlap magenta disagreement overlay."
    gt_fg = _to_numpy(gt_fg).squeeze() > 0.5
    pr_fg = _to_numpy(pr_fg).squeeze() > 0.5
    overlay = np.zeros((*gt_fg.shape, 3), np.float32)
    overlay[gt_fg & ~pr_fg] = _FN_COLOR
    overlay[pr_fg & ~gt_fg] = _FP_COLOR
    overlay[gt_fg & pr_fg] = _TP_COLOR
    ax.imshow(_disp_channel(image), cmap='gray'); ax.imshow(overlay, alpha=alpha); ax.set_axis_off()
    return ax

def _one_hot(labels, num_classes: int) -> np.ndarray:
    "(H, W) label map -> (num_classes, H, W) 0/1 stack; labels outside [0, num_classes) (e.g. 255 ignore) belong to no class."
    labels = labels.astype(int)
    return (labels[None] == np.arange(num_classes)[:, None, None]).astype(int)

def _poster_masks(logits: torch.Tensor, mask, task, class_idx, threshold) -> tuple:
    "GT/pred masks for one poster row: (H, W) binary, or (K, H, W) 0/1 stacks for per-channel rendering."
    ch = logits.detach().squeeze(0).float().cpu()  # (C, H, W); .numpy() below needs fp32 host memory
    m = _to_numpy(mask).squeeze()                # (H, W) or (K, H, W)
    if class_idx is not None:                    # one multilabel class head -> binary panels
        gt = (m[class_idx] > 0.5).astype(int) if m.ndim == 3 else (m > 0.5).astype(int)
        pr = (torch.sigmoid(ch[class_idx]) >= threshold).numpy().astype(int)
        return gt, pr
    if task == 'multilabel':
        gt = (m > 0.5).astype(int) if m.ndim == 3 else _one_hot(m, ch.shape[0])
        pr = (torch.sigmoid(ch) >= threshold).numpy().astype(int)
        return gt, pr
    if ch.shape[0] == 1:                         # binary
        return (m > 0.5).astype(int), (torch.sigmoid(ch[0]) >= threshold).numpy().astype(int)
    gt = _one_hot(m, ch.shape[0])                # multiclass: label maps -> one-hot stacks
    pr = _one_hot(ch.argmax(0).numpy(), ch.shape[0]) * gt.any(0)   # ignored pixels carry no prediction either
    return gt, pr

@torch.no_grad()
def save_error_poster(
    model, dataset, ranked, device, path,
    rank_by: str = 'score',     # shown in row titles (e.g. 'loss' or 'iou')
    k: int = 12,                # rows: the top-k ranked samples
    alpha: float = 0.1,         # GT / pred overlay transparency
    threshold: float = 0.5,     # sigmoid threshold for prediction masks
    jpeg_quality: int = 90,
    task: str = None,           # 'binary' | 'multiclass' | 'multilabel'; None infers from channel count
    class_idx: int = None,      # render ONE multilabel class head (binary panels)
    class_names=None,           # K legend labels for per-channel rendering
) -> None:
    """Save a 4-column error poster (image | GT overlay | pred overlay | difference) for the top-k ranked samples.

    One row per sample; the difference column colors FN green, FP red, overlap magenta.
    Per-channel rows (multilabel without class_idx) get a class legend.
    """
    top = ranked[:k]
    if not top:
        return
    fig, axes = plt.subplots(len(top), 4, figsize=(4 * 3, len(top) * 3))
    axes = np.atleast_2d(axes)
    try:
        with _eval_mode(model):
            for i, rec in enumerate(top):
                ax_img, ax_gt, ax_pr, ax_df = axes[i]
                s = dataset[rec['idx']]
                logits = model(s['image'].unsqueeze(0).to(device))
                gt, pr = _poster_masks(logits, s['mask'], task, class_idx, threshold)
                ax_img.imshow(_disp_channel(s['image']), cmap='gray'); ax_img.set_axis_off()
                ax_img.set_title(f"{rec['sample_id']}  {rank_by}={rec['score']:.4g}", fontsize=7)
                if gt.ndim == 3:
                    overlay_multilabel(s['image'], gt, class_names, alpha=alpha, ax=ax_gt)
                    overlay_multilabel(s['image'], pr, class_names, alpha=alpha, ax=ax_pr)
                else:
                    _flat_overlay(s['image'], gt, (0.0, 0.8, 0.0), alpha, ax=ax_gt)
                    _flat_overlay(s['image'], pr, (0.8, 0.0, 0.0), alpha, ax=ax_pr)
                _diff_overlay(s['image'], gt.any(0) if gt.ndim == 3 else gt,
                              pr.any(0) if pr.ndim == 3 else pr, alpha, ax=ax_df)
        for j, name in enumerate(['image', 'GT', 'pred', 'diff']):
            axes[0, j].set_xlabel(name, fontsize=7)
        fig.legend(handles=[mpatches.Patch(color=_FN_COLOR, label='FN (miss)'),
                            mpatches.Patch(color=_FP_COLOR, label='FP (false alarm)'),
                            mpatches.Patch(color=_TP_COLOR, label='TP (overlap)')],
                   loc='lower center', ncol=3, fontsize=7)
        plt.tight_layout(rect=(0, 0.03, 1, 1))
        fig.savefig(path, format='jpeg', dpi=100, pil_kwargs={'quality': jpeg_quality}, bbox_inches='tight')
    finally:
        plt.close(fig)

In [ ]:
#| test
# test: rank_samples / rank_samples_per_class / save_error_poster on synthetic multilabel + binary data (via exports)
import tempfile
import torch.nn as nn

r_loss = rank_samples(ml_model, toy, crit, dev, rank_by='loss')
r_iou = rank_samples(ml_model, toy, crit, dev, rank_by='iou', task='multilabel')
assert len(r_loss) == len(toy) and r_loss[0]['score'] >= r_loss[-1]['score']
assert len(r_iou) == len(toy) and r_iou[0]['score'] <= r_iou[-1]['score']
assert {r['idx'] for r in r_iou} == set(range(len(toy)))
r_c = rank_samples_per_class(ml_model, toy, crit, dev, class_idx=1, rank_by='loss')
assert len(r_c) == len(toy) and r_c[0]['score'] >= r_c[-1]['score']
try:
    rank_samples(ml_model, toy, crit, dev, rank_by='f1')
    raise RuntimeError('bad rank_by should have failed')
except AssertionError:
    pass

class _ToyBin(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:10, 4:10] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': f'bin_{i}'}

bin_ds = _ToyBin()
bin_model = nn.Conv2d(3, 1, 3, padding=1)
r_bin = rank_samples(bin_model, bin_ds, SegLoss(), dev, rank_by='dice')
assert len(r_bin) == 4 and r_bin[0]['score'] <= r_bin[-1]['score']

with tempfile.TemporaryDirectory() as d:
    save_error_poster(ml_model, toy, r_iou, dev, Path(d) / 'val_top3.jpg',
                      rank_by='iou', k=3, task='multilabel', class_names=['chip', 'void', 'lifted'])
    assert (Path(d) / 'val_top3.jpg').exists()
    save_error_poster(ml_model, toy, r_c, dev, Path(d) / 'val_top3_class1.jpg',
                      rank_by='loss', k=3, task='multilabel', class_idx=1)
    assert (Path(d) / 'val_top3_class1.jpg').exists()
    save_error_poster(bin_model, bin_ds, r_bin, dev, Path(d) / 'val_top2.jpg',
                      rank_by='dice', k=2)
    assert (Path(d) / 'val_top2.jpg').exists()
print('error poster exports ✓ worst-first ranking + 4-column JPEGs (multilabel, per-class, binary)')

In [ ]:
#| test
# test: save_error_poster must work when the model lives on CUDA (issue #69); logits are moved to CPU before .numpy()
if torch.cuda.is_available():
    import copy
    cuda = torch.device('cuda')
    ml_gpu, bin_gpu = copy.deepcopy(ml_model).to(cuda), copy.deepcopy(bin_model).to(cuda)   # copies: shared fixtures are never moved (issue #73)
    with tempfile.TemporaryDirectory() as d:
        save_error_poster(ml_gpu, toy, r_iou, cuda, Path(d) / 'ml_cuda.jpg', k=2, task='multilabel')
        save_error_poster(ml_gpu, toy, r_c, cuda, Path(d) / 'cls_cuda.jpg', k=2, task='multilabel', class_idx=1)
        save_error_poster(bin_gpu, bin_ds, r_bin, cuda, Path(d) / 'bin_cuda.jpg', k=2)
        mc_logits = torch.randn(1, 3, 16, 16, device=cuda)
        gt_mc, pr_mc = _poster_masks(mc_logits, torch.randint(0, 3, (16, 16)), 'multiclass', None, 0.5)
        assert gt_mc.shape == pr_mc.shape == (3, 16, 16)
        assert all((Path(d) / f).exists() for f in ('ml_cuda.jpg', 'cls_cuda.jpg', 'bin_cuda.jpg'))
    assert not next(ml_model.parameters()).is_cuda and not next(bin_model.parameters()).is_cuda, 'fixtures must stay on CPU'
    print('error poster on CUDA model ✓ (multilabel, per-class, binary, multiclass)')
else:
    print('error poster CUDA test skipped (no GPU)')


In [ ]:
#| test
# test: _poster_masks leaves the device and upcasts before numpy/threshold -- runs on CPU, no GPU needed (issue #73)
class _DeviceLogits:
    "Stands in for CUDA logits: only the calls `_poster_masks` may make before `.cpu()` are allowed, so dropping `.cpu()` fails here."
    def __init__(self, t): self._t = t
    def detach(self): return _DeviceLogits(self._t.detach())
    def squeeze(self, *a): return _DeviceLogits(self._t.squeeze(*a))
    def float(self): return _DeviceLogits(self._t.float())
    def cpu(self): return self._t
    def __getattr__(self, name): raise AssertionError(f'.{name} used on device logits before .cpu()')

g_ = torch.Generator().manual_seed(0)
cases = {   # name -> (channels, mask, task, class_idx)
    'binary':     (1, (torch.rand(8, 8, generator=g_) > 0.5).long(), 'binary', None),
    'multiclass': (3, torch.randint(0, 3, (8, 8), generator=g_), 'multiclass', None),
    'multilabel': (3, (torch.rand(3, 8, 8, generator=g_) > 0.5).float(), 'multilabel', None),
    'class_idx':  (3, (torch.rand(3, 8, 8, generator=g_) > 0.5).float(), 'multilabel', 1),
}
for name, (c_, mask_, task_, ci_) in cases.items():
    lg_ = torch.randn(1, c_, 8, 8, generator=g_)
    for dt_ in (torch.float32, torch.float16, torch.bfloat16):
        lg_dt = lg_.to(dt_)
        want = _poster_masks(lg_dt.float(), mask_, task_, ci_, 0.5)
        got = _poster_masks(_DeviceLogits(lg_dt), mask_, task_, ci_, 0.5)
        assert all(isinstance(o, np.ndarray) and np.array_equal(w, o) for w, o in zip(want, got)), (name, dt_)

edge = torch.full((1, 1, 4, 4), -0.001).to(torch.bfloat16)   # sigmoid = 0.49975 < 0.5, but bf16 rounds it to 0.5
_, pr = _poster_masks(edge, torch.zeros(4, 4), 'binary', None, 0.5)
assert pr.sum() == 0, 'half-precision logits must be upcast before sigmoid/threshold'
print('poster masks: device + fp16/bf16 handling (CPU-runnable) ✓')


In [ ]:
#| test
# test: ranking/poster helpers leave model.training untouched and never leak figures (issue #71)
import matplotlib.pyplot as plt
for m_ in (ml_model, bin_model): m_.train()
with tempfile.TemporaryDirectory() as d:
    rank_samples(ml_model, toy, crit, dev, rank_by='loss', task='multilabel')
    rank_samples_per_class(ml_model, toy, crit, dev, class_idx=1, rank_by='loss')
    save_error_poster(ml_model, toy, r_iou, dev, Path(d) / 'm.jpg', k=2, task='multilabel')
    assert ml_model.training, 'helpers must restore train mode'
    ml_model.eval()
    rank_samples(ml_model, toy, crit, dev, rank_by='loss', task='multilabel')
    assert not ml_model.training, 'eval mode must stay eval'

    class _Boom(nn.Module):
        training_seen = None
        def forward(self, x): raise RuntimeError('boom')
    boom = _Boom().train()
    n_figs = len(plt.get_fignums())
    try:
        save_error_poster(boom, bin_ds, r_bin, dev, Path(d) / 'boom.jpg', k=2)
        raise AssertionError('expected RuntimeError')
    except RuntimeError:
        pass
    assert len(plt.get_fignums()) == n_figs, 'figure leaked after a rendering error'
    assert boom.training, 'train mode must be restored even when the call raises'
    try: rank_samples(boom, bin_ds, SegLoss(), dev)
    except RuntimeError: pass
    assert boom.training
bin_model.eval()
print('train/eval mode restored + no figure leak on error ✓')


In [ ]:
#| test
# test: restoring mode is per-module, so a frozen (eval) BatchNorm inside a training model stays frozen (#71, EncoderDriftCheck contract)
frozen = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.BatchNorm2d(4), nn.Conv2d(4, 1, 1))
frozen.train(); frozen[1].eval()
with _eval_mode(frozen):
    assert not any(m_.training for m_ in frozen.modules())
assert frozen.training and frozen[0].training and frozen[2].training
assert not frozen[1].training, 'frozen BatchNorm must stay in eval mode'
bn_before = frozen[1].running_mean.clone()
rank_samples(frozen, bin_ds, SegLoss(), dev)
compute_patch_losses(frozen, bin_ds, SegLoss(), dev)
assert not frozen[1].training and torch.equal(bn_before, frozen[1].running_mean)
print('per-module train/eval flags restored (frozen BN stays frozen) ✓')


In [ ]:
#| test
# test: _poster_masks tolerates ignore labels and sizes GT from the model's channel count (issue #72)
g_ = torch.manual_seed(0)
lg = torch.randn(1, 3, 8, 8)
lab = torch.randint(0, 3, (8, 8)); lab[0, :4] = 255                # multiclass with an ignore index
gt, pr = _poster_masks(lg, lab, 'multiclass', None, 0.5)
assert gt.shape == pr.shape == (3, 8, 8)
assert gt[:, 0, :4].sum() == 0, 'ignored pixels belong to no class'
assert (gt.sum(0)[1:] == 1).all() and (gt.sum(0)[0, 4:] == 1).all()
assert pr[:, 0, :4].sum() == 0, 'ignored pixels must not show up as predictions (would paint false alarms in the diff)'
assert (pr.sum(0)[1:] == 1).all()

lab2 = torch.randint(0, 2, (8, 8))                                 # 2-D label map that never reaches class 2
gt, pr = _poster_masks(lg, lab2, 'multilabel', None, 0.5)
assert gt.shape == pr.shape == (3, 8, 8), 'GT channels must follow the model, not the sample max'
assert gt[2].sum() == 0
print('poster masks: ignore labels + fixed channel count ✓')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()